# Prepare per-station ISMN metadata + time series files

<br>

For each ISMN network in `ismn_data/`, builds a `station_data/` folder with, per station and
per depth:

- `{station_ID}_metadata_{depth}.csv` -- single-station, single-depth
  metadata from that network's `stations.geojson`
- `{station_ID}_ts_{depth}.csv` -- that station's SM time series at that
  depth, from the matching `sm_*.csv` (already quality-filtered to
  `soil_moisture_flag == 'G'` by [`01a_download_ismn_data.ipynb`](01a_download_ismn_data.ipynb))

`{depth}` is `depth_from` rounded to 2 decimals (e.g. `0.05`, `0.20`). These
files feed the SM-estimation and station-overview notebooks (`03a`, `03b`,
`04`), which interpolate the time series to each SAR acquisition's
timestamp.

:::{note} Design decisions:
- **Depth rounded to 2 decimals** -- some sensors report e.g. `8 inches = 0.2032 cm`;
- **`Land_cover` is decoded from `lc_2010_val`** via a hardcoded ESA CCI
  Land Cover legend -- `stations.geojson` only stores the numeric code.
:::

<hr>

## Overview

1. Configuration: networks, metadata field order, land-cover legend
2. Helper functions: match a station to its sm_*.csv file(s) (one per depth), decode land cover
3. Process each network: build and write both files per station per depth

<hr>

## 1. Configuration

`METADATA_FIELDS` fixes the row order of each `{station_ID}_metadata.csv` (a
headerless two-column `field,value` file). `CCI_LC_LEGEND` maps the numeric
`lc_2010_val` code (ESA CCI Land Cover 300 m classification, code list v2.0.7) to its
class name.

In [1]:
import csv
import json
import re
from pathlib import Path

import pandas as pd

ISMN_ROOT = Path.cwd() / "ismn_data"
NETWORKS = ["alaska"]

MISSING = -999

METADATA_FIELDS = [
    "station_ID",
    "timerange_from",
    "timerange_to",
    "depth_from",
    "depth_to",
    "longitude_val",
    "latitude_val",
    "instrument_val",
    "elevation_val",
    "network_val",
    "clay_fraction_val",
    "clay_fraction_depth_from",
    "clay_fraction_depth_to",
    "sand_fraction_val",
    "sand_fraction_depth_from",
    "sand_fraction_depth_to",
    "silt_fraction_val",
    "silt_fraction_depth_from",
    "silt_fraction_depth_to",
    "climate_KG_val",
    "Land_cover",
    "organic_carbon_val",
    "organic_carbon_depth_from",
    "organic_carbon_depth_to",
    "saturation_val",
    "saturation_depth_from",
    "saturation_depth_to",
]

CCI_LC_LEGEND = {
    "0": "No data",
    "10": "Cropland, rainfed",
    "11": "Cropland, rainfed, herbaceous cover",
    "12": "Cropland, rainfed, tree or shrub cover",
    "20": "Cropland, irrigated or post-flooding",
    "30": "Mosaic cropland (>50%) / natural vegetation (<50%)",
    "40": "Mosaic natural vegetation (>50%) / cropland (<50%)",
    "50": "Tree cover, broadleaved, evergreen, closed to open (>15%)",
    "60": "Tree cover, broadleaved, deciduous, closed to open (>15%)",
    "61": "Tree cover, broadleaved, deciduous, closed (>40%)",
    "62": "Tree cover, broadleaved, deciduous, open (15-40%)",
    "70": "Tree cover, needleleaved, evergreen, closed to open (>15%)",
    "71": "Tree cover, needleleaved, evergreen, closed (>40%)",
    "72": "Tree cover, needleleaved, evergreen, open (15-40%)",
    "80": "Tree cover, needleleaved, deciduous, closed to open (>15%)",
    "81": "Tree cover, needleleaved, deciduous, closed (>40%)",
    "82": "Tree cover, needleleaved, deciduous, open (15-40%)",
    "90": "Tree cover, mixed leaf type",
    "100": "Mosaic tree and shrub (>50%) / herbaceous cover (<50%)",
    "110": "Mosaic herbaceous cover (>50%) / tree and shrub (<50%)",
    "120": "Shrubland",
    "121": "Shrubland, evergreen",
    "122": "Shrubland, deciduous",
    "130": "Grassland",
    "140": "Lichens and mosses",
    "150": "Sparse vegetation (<15%)",
    "151": "Sparse tree (<15%)",
    "152": "Sparse shrub (<15%)",
    "153": "Sparse herbaceous cover (<15%)",
    "160": "Tree cover, flooded, fresh or brackish water",
    "170": "Tree cover, flooded, saline water",
    "180": "Shrub or herbaceous cover, flooded, fresh/saline/brackish water",
    "190": "Urban areas",
    "200": "Bare areas",
    "201": "Consolidated bare areas",
    "202": "Unconsolidated bare areas",
    "210": "Water bodies",
    "220": "Permanent snow and ice",
}

## 2. Helper functions

`find_checked_csvs` returns every `sm_*.csv` for a station (written
directly into the network's own directory by `01a_download_ismn_data.ipynb`)
-- one per sensor depth, parsing `depth_from`/`depth_to` back out of the
filename and rounding to 2 decimals. If none match, the station has no data
file at all and is skipped in step 3.

In [2]:
def find_checked_csvs(checked_dir: Path, station_id: str):
    pattern = re.compile(rf"^sm_{re.escape(station_id)}_.+_([0-9.]+)_([0-9.]+)\.csv$")
    matches = []
    for p in checked_dir.glob("sm_*.csv"):
        m = pattern.match(p.name)
        if m:
            depth_from, depth_to = m.groups()
            matches.append((p, round(float(depth_from), 2), round(float(depth_to), 2)))
    return matches


def decode_land_cover(lc_code) -> str:
    if lc_code is None or lc_code == "":
        return MISSING
    return CCI_LC_LEGEND.get(str(lc_code).strip(), MISSING)


def get_prop(properties: dict, key: str):
    value = properties.get(key)
    return MISSING if value is None or value == "" else value

## 3. Process each network

For each network: create `station_data/`, load `stations.geojson`, and for every
station -- match its `checked/` CSV, derive `timerange_from`/`timerange_to` from the
actual data, build the metadata row, and write both output files.

In [ ]:
for network in NETWORKS:
    network_dir = ISMN_ROOT / network
    # 01a_download_ismn_data.ipynb writes sm_*.csv files directly into
    # network_dir (no "checked/" subfolder is ever created).
    checked_dir = network_dir
    output_dir = network_dir / "station_data"
    output_dir.mkdir(parents=True, exist_ok=True)

    with open(network_dir / "stations.geojson") as f:
        stations_geojson = json.load(f)

    n_written = 0
    n_stations = len(stations_geojson["features"])
    for feature in stations_geojson["features"]:
        props = feature["properties"]
        station_id = props.get("station_val", props.get("index"))

        matches = find_checked_csvs(checked_dir, station_id)
        if not matches:
            print(f"  skipping {station_id}: no matching sm_*.csv found for this station")
            continue

        # One (ts.csv, metadata.csv) pair per sensor depth -- a station can
        # report more than one (see find_checked_csvs above), and depth_from/
        # depth_to genuinely differ between them, so a single merged metadata
        # file could only correctly describe one depth at a time anyway.
        for ts_path, depth_from, depth_to in matches:
            ts_df = pd.read_csv(ts_path)
            if ts_df.empty:
                print(f"  skipping {station_id} ({depth_from:.2f}m): {ts_path.name} has no data rows")
                continue

            ts_df["date_time"] = pd.to_datetime(ts_df["date_time"])
            ts_df = ts_df.sort_values("date_time")

            depth_str = f"{depth_from:.2f}"

            metadata_row = {
                "station_ID": station_id,
                "timerange_from": ts_df["date_time"].min().strftime("%Y-%m-%d %H:%M:%S"),
                "timerange_to": ts_df["date_time"].max().strftime("%Y-%m-%d %H:%M:%S"),
                "depth_from": depth_str,
                "depth_to": f"{depth_to:.2f}",
                "longitude_val": get_prop(props, "longitude_val"),
                "latitude_val": get_prop(props, "latitude_val"),
                "instrument_val": get_prop(props, "instrument_val"),
                "elevation_val": get_prop(props, "elevation_val"),
                "network_val": get_prop(props, "network_val"),
                "clay_fraction_val": get_prop(props, "clay_fraction_val"),
                "clay_fraction_depth_from": get_prop(props, "clay_fraction_depth_from"),
                "clay_fraction_depth_to": get_prop(props, "clay_fraction_depth_to"),
                "sand_fraction_val": get_prop(props, "sand_fraction_val"),
                "sand_fraction_depth_from": get_prop(props, "sand_fraction_depth_from"),
                "sand_fraction_depth_to": get_prop(props, "sand_fraction_depth_to"),
                "silt_fraction_val": get_prop(props, "silt_fraction_val"),
                "silt_fraction_depth_from": get_prop(props, "silt_fraction_depth_from"),
                "silt_fraction_depth_to": get_prop(props, "silt_fraction_depth_to"),
                "climate_KG_val": get_prop(props, "climate_KG_val"),
                "Land_cover": decode_land_cover(props.get("lc_2010_val")),
                "organic_carbon_val": get_prop(props, "organic_carbon_val"),
                "organic_carbon_depth_from": get_prop(props, "organic_carbon_depth_from"),
                "organic_carbon_depth_to": get_prop(props, "organic_carbon_depth_to"),
                "saturation_val": get_prop(props, "saturation_val"),
                "saturation_depth_from": get_prop(props, "saturation_depth_from"),
                "saturation_depth_to": get_prop(props, "saturation_depth_to"),
            }

            with open(output_dir / f"{station_id}_metadata_{depth_str}.csv", "w", newline="") as f:
                writer = csv.writer(f)
                for field in METADATA_FIELDS:
                    writer.writerow([field, metadata_row[field]])

            ts_out = ts_df.rename(
                columns={"date_time": "datetime_UTC", "soil_moisture": f"sm_{depth_str}"}
            )
            ts_out.to_csv(output_dir / f"{station_id}_ts_{depth_str}.csv", index=False)

            n_written += 1

    print(f"{network}: wrote station_data/ for {n_written} station-depth file(s) "
          f"across {n_stations} station(s) -> {output_dir}")

  skipping TurnagainPass (0.05m): sm_TurnagainPass_Stevens-Hydraprobe-Digital_1_1_0.0508_0.0508.csv has no data rows
  skipping TurnagainPass (0.20m): sm_TurnagainPass_Stevens-Hydraprobe-Digital_1_1_0.2032_0.2032.csv has no data rows
alaska: wrote station_data/ for 37 station-depth file(s) across 20 station(s) -> D:\Docs\RS_2023\NISAR4SM\ismn_data\alaska\station_data


<hr>

## Summary

Each network directory under `ismn_data/` now has a `station_data/` folder
with one `{station_ID}_metadata_{depth}.csv` and one
`{station_ID}_ts_{depth}.csv` per station per reported sensor depth, ready
to be read by the SM-estimation and station-overview notebooks (`03a`/`03b`/`04`).

<hr>